# v10a.20b recovery cell

**Use only in the same Colab runtime that just failed at `heapq.heappush(danger, rec)`.** It resumes at section `[9]` without rebuilding sections `[3]`–`[8]`.

In [ ]:
# HODGE v10a.20b — RESUME FROM THE EXACT POINT OF THE heapq FAILURE
# Run this in a NEW Colab cell in the SAME runtime/kernel that produced the
# NameError.  Do not restart the runtime first.
import heapq

print('\n[9] ARITHMETIC DENOMINATOR-LIFT HAAR EXACTIFICATION')

LIFT_TOL=float(os.environ.get('V10A20_LIFT_TOL','1e-5'))
DENSE_SAMPLE=int(os.environ.get('V10A20_DENSE_SAMPLE','384'))
LONGDOUBLE_TOP=int(os.environ.get('V10A20_LONGDOUBLE_TOP','256'))
DEBUG_LIMIT=int(os.environ.get('V10A20_DEBUG_PAIR_LIMIT','0'))

# Fast endpoint signature without re-canonicalizing an already canonical pair.
def _x_sig_fast(a,b):
    return tuple(sorted(_x_local_patterns_fast(a,b)))

# Universal bound follows from 24 total occurrences and
# q(pattern) <= 120^(occurrences/6) for every supported local pattern.
QH_UNIVERSAL=120**4
gate('v10a.20 universal two-step Haar denominator bound q_H<=120^4',
     QH_UNIVERSAL==207360000,QH_UNIVERSAL)

# Long-double version of the same low-rank factorized contractor.  This is used
# only on the most numerically dangerous topologies.
def _x_ld_factor_data():
    if hasattr(_x_ld_factor_data,'data'):
        return _x_ld_factor_data.data
    fd=_v10a13_factor_data()
    out={}
    for k in (1,2,3):
        D,W=fd[('bal',k)]
        out[('bal',k)]=(np.asarray(D,dtype=np.longdouble),
                        np.asarray(W,dtype=np.longdouble))
    I,C=fd['six']
    out['six']=(np.asarray(I,dtype=np.longdouble),
                np.asarray(C,dtype=np.longdouble))
    out['eps']=np.asarray(_FAST_EPS,dtype=np.longdouble)
    _x_ld_factor_data.data=out
    return out

def _x_haar_longdouble(a,b):
    occ,part=lx_combine_bra_ket(a,b)
    by=defaultdict(lambda:{True:[],False:[]})
    for i,(l,t) in enumerate(occ):
        by[int(l)][bool(t)].append(i)
    data=_x_ld_factor_data()
    args=[]
    pref=np.longdouble(1)
    aux=(max(part)+1) if part else 0
    for g in by.values():
        U,B=g[True],g[False]
        pat=(len(U),len(B))
        if (pat[0]-pat[1])%3:
            return np.longdouble(0)
        if pat in ((1,1),(2,2),(3,3)):
            k=len(U)
            D,W=data[('bal',k)]
            ar=aux; ac=aux+1; aux+=2
            row=[int(part[2*p]) for p in U+B]
            col=[int(part[2*p+1]) for p in U+B]
            args.extend((D,[ar]+row,W,[ar,ac],D,[ac]+col))
        elif pat in ((3,0),(0,3)):
            items=U if len(U)==3 else B
            row=[int(part[2*p]) for p in items]
            col=[int(part[2*p+1]) for p in items]
            E=data['eps']
            args.extend((E,row,E,col))
            pref/=np.longdouble(6)
        elif pat in ((6,0),(0,6)):
            items=U if len(U)==6 else B
            I,C=data['six']
            ar=aux; ac=aux+1; aux+=2
            row=[int(part[2*p]) for p in items]
            col=[int(part[2*p+1]) for p in items]
            args.extend((I,[ar]+row,C,[ar,ac],I,[ac]+col))
        else:
            raise RuntimeError(f'v10a.20 unsupported longdouble Haar pattern {pat}')
    if not args:
        return np.longdouble(N)**len(set(part))
    return pref*oe.contract(*args,[],optimize='greedy',memory_limit='max_input')

items=list(pairw.items())
items.sort(key=lambda kv:_v10a13_pair_score(kv[0][0],kv[0][1]),reverse=True)
full_n=len(items)
if DEBUG_LIMIT>0:
    items=items[:DEBUG_LIMIT]
    print(f'  DEBUG: lifting only first {len(items)} of {full_n} exact topologies')

# Independent dense sample is deterministic and stratified through the sorted
# complexity list.  Always include endpoint-signature representatives.
sample_idx=set()
if items:
    ns=min(DENSE_SAMPLE,len(items))
    if ns==1:
        sample_idx.add(0)
    else:
        for j in range(ns):
            sample_idx.add(round(j*(len(items)-1)/(ns-1)))
reps={}
for i,((a,b),w) in enumerate(items):
    reps.setdefault(_x_sig_fast(a,b),i)
sample_idx.update(reps.values())

# Accumulate D exactly over the previously proven global denominator divisor QBOUND.
D11=_XQ(-13,896)
if QBOUND % D11.denominator:
    raise RuntimeError('QBOUND does not contain the analytic one-face denominator')
TOTAL_NUM=D11.numerator*(QBOUND//D11.denominator)

max_res=0.0
max_qh=0
nonzero_haar=0
dense_checked=0
dense_maxdiff=0.0
danger=[]  # min-heap of the LONGDOUBLE_TOP largest lift residuals

t2=time.time(); last=t2
for ii,((a,b),w) in enumerate(items,1):
    qh=_x_haar_den_bound(a,b)
    if qh>QH_UNIVERSAL:
        raise RuntimeError(f'Haar denominator bound escaped universal 120^4: {qh}')
    max_qh=max(max_qh,qh)

    hf=float(_v10a13_haar_factor.__wrapped__(a,b))
    yf=hf*qh
    nh=int(round(yf))
    resid=abs(yf-nh)
    max_res=max(max_res,resid)

    if resid>=LIFT_TOL:
        raise RuntimeError(
            f'Haar integer lift too close to ambiguity at topology {ii}: '
            f'qH={qh}, h={hf:.17g}, qH*h={yf:.17g}, nearest={nh}, residual={resid:.3e}'
        )

    # Exact lifted Haar rational is nh/qh.  Keep the unreduced qh because the
    # global denominator divisibility proof is expressed in these common local
    # projector denominators.
    if nh:
        nonzero_haar+=1

    termden=2*w.denominator*qh
    if QBOUND % termden:
        raise RuntimeError(
            f'rigorous QBOUND missed a term denominator: termden={termden}, qh={qh}, w={w}'
        )
    TOTAL_NUM += w.numerator*nh*(QBOUND//termden)

    # Independent dense contractor on a deterministic full-complexity sample.
    if (ii-1) in sample_idx:
        hd=float(_qcache(a,b))
        nd=int(round(hd*qh))
        dense_checked+=1
        dense_maxdiff=max(dense_maxdiff,abs(hd-hf))
        if nd!=nh:
            raise RuntimeError(
                f'dense/factorized integer lift mismatch topology={ii}: '
                f'factor n={nh}, dense n={nd}, qH={qh}, hf={hf}, hd={hd}'
            )

    # Preserve only the largest residuals for independent long-double replay.
    rec=(resid,ii,a,b,qh,nh,hf)
    if LONGDOUBLE_TOP>0:
        if len(danger)<LONGDOUBLE_TOP:
            heapq.heappush(danger,rec)
        elif resid>danger[0][0]:
            heapq.heapreplace(danger,rec)

    now=time.time()
    if V10A7_PROGRESS and (now-last>=XHEART or ii%1000==0 or ii==len(items)):
        rate=ii/max(now-t2,1e-9)
        eta=(len(items)-ii)/rate if rate else 0.0
        print(f'      exact Haar lift {ii:,}/{len(items):,}; '
              f'rate={rate:,.1f}/s; max(qH)={max_qh:,}; '
              f'max lift residual={max_res:.3e}; dense={dense_checked:,}; '
              f'elapsed={now-t2:.1f}s ETA~{eta:.1f}s; RAM={_v10a8_mem_gb():.2f} GiB',
              flush=True)
        last=now

gate('v10a.20 every factorized Haar topology is safely integer-lifted',
     max_res<LIFT_TOL,f'max residual={max_res:.3e}, qHmax={max_qh}')
gate('v10a.20 independent dense contractor gives identical lifted integers',
     dense_checked==len(sample_idx) and dense_maxdiff<5e-12,
     f'checked={dense_checked}, max float diff={dense_maxdiff:.3e}')

# Long-double replay of the top residuals: these are the cases closest to a
# hypothetical half-integer ambiguity, so they are the most valuable precision check.
ld_bad=[]
ld_maxdiff=np.longdouble(0)
for resid,ii,a,b,qh,nh,hf in sorted(danger,reverse=True):
    hld=_x_haar_longdouble(a,b)
    nld=int(np.rint(hld*np.longdouble(qh)))
    ld_maxdiff=max(ld_maxdiff,abs(hld-np.longdouble(hf)))
    if nld!=nh:
        ld_bad.append((ii,qh,nh,nld,float(hf),str(hld)))
        if len(ld_bad)>=5:
            break
gate('v10a.20 long-double replay confirms the closest-to-ambiguity integer lifts',
     len(ld_bad)==0,
     f'checked={len(danger)}, max |longdouble-f64|={float(ld_maxdiff):.3e}'
     if not ld_bad else ld_bad[0])

if DEBUG_LIMIT>0:
    print('DEBUG denominator-lift contractor completed successfully; no final D_A emitted from a truncated corpus.')
    raise SystemExit(0)

D_EXACT=_XQ(TOTAL_NUM,QBOUND)
gate('v10a.20 exact D_A denominator divides rigorous QBOUND',
     QBOUND%D_EXACT.denominator==0,D_EXACT.denominator)
gate('v10a.20 exact D_A denominator primes satisfy v10a.18 localization',
     _x_den_primes(D_EXACT)<=S4_PRIMES,sorted(_x_den_primes(D_EXACT)))

# The previous v10a.16 result is used only after the exact integer accumulation.
D_PREV=-49.7901704444838
gate('v10a.20 exact D_A agrees with the blind v10a.16 float',
     abs(float(D_EXACT)-D_PREV)<5e-12,
     f'exact={float(D_EXACT):+.15g}, prior={D_PREV:+.15g}')

FOLD_EX=_XQ(5315003,140454)
VLINK_EX=_XQ(-1474623,1675520)
M4_EXACT=D_EXACT+FOLD_EX-VLINK_EX
M4_PREV=-11.068479463777946
gate('v10a.20 exact m4_rest agrees with the blind v10a.16 float',
     abs(float(M4_EXACT)-M4_PREV)<5e-12,
     f'exact={float(M4_EXACT):+.15g}, prior={M4_PREV:+.15g}')
gate('v10a.20 exact m4 denominator primes satisfy arithmetic localization',
     _x_den_primes(M4_EXACT)<=S4_PRIMES,sorted(_x_den_primes(M4_EXACT)))

print('\n[10] EXACT FOURTH-ORDER ARITHMETIC RESULT')
print('  exact pair topologies =',len(pairw))
print('  nonzero Haar lifts     =',nonzero_haar)
print('  max q_H                =',max_qh,'<=',QH_UNIVERSAL)
print('  max integer-lift residual =',f'{max_res:.3e}')
print('  D_A exact =',D_EXACT)
print('    D numerator factorization   =',dict(_xsp.factorint(D_EXACT.numerator)))
print('    D denominator factorization =',dict(_xsp.factorint(D_EXACT.denominator)))
print('    D decimal =',repr(float(D_EXACT)))
print('  fold exact =',FOLD_EX)
print('  linked vacuum exact =',VLINK_EX)
print('  m4_rest exact =',M4_EXACT)
print('    m4 numerator factorization   =',dict(_xsp.factorint(M4_EXACT.numerator)))
print('    m4 denominator factorization =',dict(_xsp.factorint(M4_EXACT.denominator)))
print('    m4 decimal =',repr(float(M4_EXACT)))

print('\n  MASS SERIES')
print('    m0 = 8/3')
print('    m1 = 1')
print('    m2 = 11/306')
print('    m3 = -109151/249696')
print('    m4_rest =',M4_EXACT)

print('\n'+'='*132)
print('FINAL v10a.20 GATE SUMMARY')
print('='*132)
newg=gates[V10A7_GATE_START:]
for i,(name,ok,detail) in enumerate(newg,1):
    print(f"{i:02d}. {'PASS' if ok else 'FAIL'} — {name}"+(f' :: {detail}' if detail else ''))
passed=sum(ok for _,ok,_ in newg)
print('-'*132)
print(f'PASSED {passed}/{len(newg)} v10a.20 GATES')
if passed!=len(newg):
    raise AssertionError('v10a.20 gate failure')

print('\nV10A.20 CONCLUSION')
print('------------------')
print('* The sqrt(2)-normalized two-step corpus was exactified coefficient-by-coefficient.')
print('* Exact arithmetic merges 10 float-split whole blocks, 40 pair occurrences, and 2 float-split Haar topologies.')
print('* Every Haar topology was lifted through its finite SU(3) projector denominator before the final sum.')
print('* The final D_A numerator was accumulated as one integer over the rigorous arithmetic QBOUND.')
print('* m4_rest is emitted from that arithmetic ledger, not from Fraction.limit_denominator().')

try:
    if _V10A16_FH_ARMED:
        faulthandler.cancel_dump_traceback_later()
except Exception:
    pass
